# 01: Telugu Handwritten Text Recognition - Data Exploration & Vocabulary Audit

This notebook explores the **IIIT-HW-Telugu** handwritten word dataset and the Telugu Unicode character set. It also demonstrates how we construct a **data-driven character transition matrix** to restrict illegal sequences during autoregressive decoding.

### Objectives:
1. **Load and analyze** label annotations (distributions of word lengths and character frequencies).
2. **Inspect Telugu Unicode categories** (vowels, consonants, vowel signs, virama, and modifiers).
3. **Demonstrate Vocabulary tokenisation** and round-trip encode/decode consistency.
4. **Analyze the data-driven transition matrix** (visualise allowed and blocked transitions).
5. **Visualize sample image tensors** directly from the DataLoader pipeline.

## 1. Setup and Imports

In [1]:
import os
import sys
import random
from collections import Counter
import unicodedata

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image, ImageDraw, ImageFont

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from src.vocab import TeluguVocab, ALL_TELUGU_CHARS, VIRAMA
from src.dataset import TeluguHTRDataset
from src.transforms import ValTransform

print("PyTorch Version:", torch.__version__)
sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'src'

## 2. Generate Synthetic Dataset (if real dataset is not downloaded yet)

To make this notebook immediately runnable and interactive, we check if the dataset files exist. If not, we generate a small synthetic set of images and labels under `data/raw/` so the pipeline can be demonstrated.

In [ ]:
def ensure_synthetic_data():
    train_dir = os.path.join("..", "data", "raw", "train")
    val_dir = os.path.join("..", "data", "raw", "val")
    
    train_ann_path = os.path.join(train_dir, "labels.txt")
    val_ann_path = os.path.join(val_dir, "labels.txt")
    
    # Sample Telugu words
    telugu_words = [
        "కాలం", "పూజ", "తెలుగు", "భారతదేశం", "అమ్మ",
        "నాన్న", "విద్యా", "ప్రగతి", "సంస్కృతి", "విశ్వవిద్యాలయం",
        "సూర్యుడు", "చంద్రుడు", "నక్షత్రం", "సముద్రం", "పర్వతం",
        "పుస్తకం", "కలం", "కాగితం", "గణితం", "విజ్ఞానం"
    ]
    
    # Check if files already exist
    if os.path.exists(train_ann_path) and os.path.exists(val_ann_path):
        print("Real or synthetic dataset already present at data/raw/")
        return train_ann_path, val_ann_path, False
        
    print("Dataset not found. Generating a small synthetic dataset for demonstration purposes...")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    
    # Generate dummy images with text drawn on them
    for folder, count, ann_path in [
        (train_dir, 100, train_ann_path),
        (val_dir, 20, val_ann_path)
    ]:
        samples = []
        for idx in range(count):
            word = random.choice(telugu_words)
            img_name = f"word_{idx:05d}.png"
            img_path = os.path.join(folder, img_name)
            
            # Create a simple image with a background color and some lines
            img = Image.new("RGB", (256, 64), color=(240, 240, 240))
            draw = ImageDraw.Draw(img)
            # Draw some simulated scribbles/lines representing handwriting
            for _ in range(5):
                x1 = random.randint(10, 200)
                y1 = random.randint(10, 50)
                x2 = x1 + random.randint(20, 50)
                y2 = y1 + random.randint(-10, 10)
                draw.line([(x1, y1), (x2, y2)], fill=(30, 30, 30), width=random.randint(1, 3))
                
            img.save(img_path)
            samples.append(f"{img_name} {word}")
            
        with open(ann_path, "w", encoding="utf-8") as f:
            f.write("\n".join(samples))
            
    print(f"Generated 100 train samples in {train_dir}")
    print(f"Generated 20 val samples in {val_dir}")
    return train_ann_path, val_ann_path, True

train_ann, val_ann, is_synthetic = ensure_synthetic_data()

## 3. Analyse Annotation Statistics

In [ ]:
# Load labels
def load_labels(ann_path):
    labels = []
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                labels.append(parts[1])
    return labels

train_labels = load_labels(train_ann)
val_labels = load_labels(val_ann)

print(f"Loaded {len(train_labels)} training labels.")
print(f"Loaded {len(val_labels)} validation labels.")

# Calculate label lengths
train_lengths = [len(label) for label in train_labels]
mean_len = np.mean(train_lengths)
max_len = np.max(train_lengths)
min_len = np.min(train_lengths)

print(f"Label length stats: Min={min_len}, Max={max_len}, Mean={mean_len:.2f}")

### Plot Length Distribution and Character Frequency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Word length histogram
sns.histplot(train_lengths, bins=range(1, max(train_lengths)+2), kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Word Label Length Distribution")
axes[0].set_xlabel("Number of Characters (Unicode codepoints)")
axes[0].set_ylabel("Count")

# Character frequency
all_chars = [char for label in train_labels for char in label]
char_counts = Counter(all_chars)
common_chars = char_counts.most_common(20)

chars_list, counts_list = zip(*common_chars)
# Unicode representations for labels since default matplotlib fonts might not render Telugu
labels_representation = [f"{c} ({ord(c):04X})" for c in chars_list]

sns.barplot(x=list(counts_list), y=labels_representation, ax=axes[1], palette="viridis")
axes[1].set_title("Top 20 Most Frequent Characters")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Character")

plt.tight_layout()
plt.show()

## 4. Unicode Categories in Telugu

Telugu characters range from Unicode point `U+0C00` to `U+0C7F`. We map them into categories to understand their role in word construction:
- **Vowels (Aachaalu)**: Independent letters like అ, ఆ, ఇ
- **Consonants (Vyanjanaalu)**: Base structures like క, గ, చ
- **Vowel Signs (Gunintaalu)**: Dependent diacritics modifying consonants, like ా (aa), ి (i)
- **Virama (Halant)**: The sign ్ which joins consonants into conjunct forms (e.g., క + ్ + క = క్క)
- **Anusvara/Modifiers**: Marks like ం (nasalizer) and ః

In [ ]:
print("Telugu Unicode codepoints detected in train:")
unique_chars = sorted(list(char_counts.keys()))
for c in unique_chars[:15]:
    # Describe the char category
    name = unicodedata.name(c, "UNKNOWN")
    print(f"  Char: '{c}'  Code: U+{ord(c):04X}  Name: {name}")
if len(unique_chars) > 15:
    print(f"  ... and {len(unique_chars) - 15} more characters.")

## 5. Vocabulary and Tokenisation

We load or build the vocabulary using `TeluguVocab` and perform round-trip validation to ensure that no letters are lost or decoded incorrectly.

In [ ]:
# Build vocab
vocab = TeluguVocab.from_annotation_files([train_ann], build_matrix=True)

# Sample word round-trip
sample_word = train_labels[0]
encoded = vocab.encode(sample_word)
decoded = vocab.decode(encoded)

print(f"Original Word  : {sample_word}")
print(f"Encoded IDs    : {encoded}")
print(f"Decoded Word   : {decoded}")
print(f"Match          : {sample_word == decoded}")

# Look at token ids mapping
for idx in encoded:
    print(f"  ID {idx:3d} -> Char '{vocab.idx2char(idx)}' (Category: {vocab.category(idx)})")

## 6. Audit the Data-Driven Transition Matrix

The data-driven matrix blocks invalid Telugu sequences. We print the matrix statistics and audit allowed category-to-category transitions.

In [ ]:
# Print transition stats from the vocab class
vocab.print_transition_stats() if hasattr(vocab, 'print_transition_stats') else print("Vocabulary Matrix Source:", vocab._matrix_source)

### Transition Grid Heatmap

Let's visualize the allowed transitions as a grid heatmap between different character classes.

In [ ]:
cat_list = ["SOS", "VOWEL", "CONSONANT", "VOWEL_SIGN", "VIRAMA", "MODIFIER", "DIGIT", "EOS"]
V = len(vocab)
grid = np.zeros((len(cat_list), len(cat_list)))

for i, prev_cat in enumerate(cat_list):
    prev_ids = [idx for idx in range(V) if vocab.category(idx) == prev_cat]
    for j, nxt_cat in enumerate(cat_list):
        nxt_ids = [idx for idx in range(V) if vocab.category(idx) == nxt_cat]
        if not prev_ids or not nxt_ids:
            continue
        
        allowed = sum(1 for p in prev_ids for n in nxt_ids if vocab.is_valid_transition(p, n))
        total = len(prev_ids) * len(nxt_ids)
        grid[i, j] = allowed / total

plt.figure(figsize=(10, 8))
sns.heatmap(grid, xticklabels=cat_list, yticklabels=cat_list, annot=True, cmap="Blues", fmt=".2f", cbar_kws={'label': 'Allowed transitions ratio'})
plt.title("Normalized Category-to-Category Allowed Transitions Matrix")
plt.xlabel("Next Category")
plt.ylabel("Previous Category")
plt.show()

## 7. DataLoader and Image Visualization

We load a small batch of processed images and labels using the `TeluguHTRDataset` and visualize the grayscale outputs alongside their ground truth strings.

In [ ]:
# Initialize Dataset
image_root = ".." # annotation contains relative path
if is_synthetic:
    # Synthetic images are stored relative to data/raw/train
    image_root = os.path.join("..", "data", "raw")

dataset = TeluguHTRDataset(
    annotation_file=train_ann,
    image_root=image_root,
    vocab=vocab,
    transform=ValTransform(),
    add_sos_eos=False
)

print(f"Dataset Size: {len(dataset)}")

# Plot a few samples
num_samples = min(4, len(dataset))
fig, axes = plt.subplots(num_samples, 1, figsize=(10, 2.5 * num_samples))
if num_samples == 1:
    axes = [axes]

for idx in range(num_samples):
    img_tensor, label_ids, label_len = dataset[idx]
    img_np = img_tensor.squeeze(0).numpy()
    label = dataset.get_label(idx)
    
    axes[idx].imshow(img_np, cmap="gray")
    axes[idx].set_title(f"Index: {idx} | Label: '{label}' | Tensor shape: {list(img_tensor.shape)}")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()